# Providing examples

Show, don't tell: put a sample input and its ideal output into the prompt.
One example is **one-shot**, several are **multi-shot**.

Useful for corner cases (sarcasm in sentiment analysis), complex output formats (a specific JSON),
an exact style or tone, and ambiguous inputs. Use multi-shot to cover several scenarios.

Best practices:

- wrap examples in XML tags: `<sample_input>`, `<ideal_output>`
- say what you are showing: "Here is an example with a sample input and an ideal output"
- cover your most common failure cases
- explain why the example output is ideal
- keep examples relevant to the task

Where to get one: take a top-scoring output from your own evaluation report.

**The example used here** was written by the lesson 04 prompt for an athlete who is *not* in
the dataset (a lactose-intolerant triathlete), then graded by the same model grader
with a score of 9. Two slips were fixed by hand (the carb total did not match the meals,
the notes named foods that are not in the plan). An example that copies a test case would leak the answer and inflate the score.

## Setup

In [1]:
from prompt_evaluator import PromptEvaluator, add_messages, chat

evaluator = PromptEvaluator(max_concurrent_tasks=3)

## Prompt

Graded on the frozen `dataset.json` from [`01_prompt_engineering`](./01_prompt_engineering.ipynb), with the same criteria, so the scores are comparable.

In [2]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

<athlete_information>
- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
</athlete_information>

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned

Here is an example with a sample input and an ideal output:
<sample_input>
- Height: 168 cm
- Weight: 60 kg
- Goal: Olympic-distance triathlon training, two sessions a day
- Dietary restrictions: Lactose intolerant
</sample_input>
<ideal_output>
# One-Day Meal Plan for Olympic-Distance Triathlon Training
**Lactose-Intolerant Athlete | 168 cm, 60 kg**

---

## Daily Nutrition Summary
- **Calories:** 3,200 kcal (for two training sessions)
- **Protein:** 145g
- **Carbohydrates:** 375g
- **Fat:** 95g

---

## Meal Schedule

### **BREAKFAST** (7:00 AM) — 650 kcal
*Before morning session*
- Oatmeal with water: 80g dry
- Banana: 150g
- Almond butter: 30g
- Honey: 15g
- Rice milk: 200ml

**Macros:** Protein 18g | Carbs 85g | Fat 20g

---

### **MORNING SNACK** (10:30 AM) — 300 kcal
*Post-swim/run, before eating lunch*
- Rice cakes: 50g
- Peanut butter: 25g
- Apple: 150g
- Water: 500ml

**Macros:** Protein 9g | Carbs 42g | Fat 12g

---

### **LUNCH** (12:30 PM) — 750 kcal
*2–3 hours before afternoon session*
- Grilled chicken breast: 150g
- White rice: 180g (cooked)
- Olive oil: 10ml
- Broccoli: 150g
- Salt & pepper to taste

**Macros:** Protein 45g | Carbs 75g | Fat 15g

---

### **PRE-TRAINING SNACK** (3:30 PM) — 250 kcal
*30–45 minutes before afternoon session*
- Energy bar (dairy-free): 1x 65g
- Mango juice (100%): 200ml

**Macros:** Protein 5g | Carbs 55g | Fat 3g

---

### **DINNER** (7:00 PM) — 950 kcal
*Post-training*
- Baked salmon: 180g
- Sweet potato: 250g (baked)
- Spinach salad: 100g
- Extra virgin olive oil: 12ml
- Lemon juice & garlic to taste

**Macros:** Protein 42g | Carbs 90g | Fat 30g

---

### **EVENING SNACK** (9:30 PM) — 300 kcal
*Recovery support*
- Turkey breast: 100g
- Gluten-free bread: 50g (2 slices)
- Avocado: 60g
- Coconut water: 200ml

**Macros:** Protein 26g | Carbs 28g | Fat 15g

---

## Hydration Plan
- **During morning session:** 500ml sports drink (electrolytes, no dairy)
- **Between sessions:** 750ml water
- **During afternoon session:** 500ml sports drink
- **Throughout day:** Minimum 2L water total

---

## Notes for Athlete
✓ Dairy-free throughout (rice milk, nut butters, no cheese or yogurt)
✓ Balanced carb-to-protein ratio for endurance + recovery
✓ Timing supports two training sessions
✓ Budget-friendly whole foods emphasis
✓ Anti-inflammatory foods (salmon, spinach, olive oil)
</ideal_output>
This example is ideal because it states the daily calorie total and the protein, fat and carb targets up front, gives every meal a time that fits around both training sessions, lists each food with a portion in grams, and avoids every lactose source.
"""
    messages = []
    add_messages(messages, prompt)
    return chat(messages)

## Evaluate

In [3]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
    json_output_file="output_05.json",
    html_output_file="output_05.html",
)

Graded 1/3 test cases


Graded 2/3 test cases


Graded 3/3 test cases
Average score: 7.666666666666667


In [4]:
for result in results:
    print(f"\n[{result['score']}] {result['test_case']['scenario']}")
    print(f"    {result['reasoning']}")


[8] Testing with a high-endurance athlete (marathon runner) requiring calorie-dense meals and carbohydrate loading strategy
    The solution comprehensively satisfies all mandatory requirements with precise caloric totals, detailed macronutrient breakdowns, and exact food portions with timing. The carbohydrate loading strategy is scientifically sound and appropriately scaled for the athlete's weight. The format is genuinely compact and suitable for quick reference. However, the caloric total sits conservatively at the lower boundary of the recommended range, and the meal structure assumes a specific training schedule not explicitly stated in inputs. These are secondary concerns that don't violate mandatory requirements but represent minor optimization opportunities.

[6] Testing with a strength-focused athlete (weightlifter) emphasizing protein intake and meal timing around workouts
    The solution comprehensively addresses all mandatory requirements with precise daily caloric totals

Full report: `output_05.html` (git-ignored).